## Simulated survey:
1. pool of R respondents each with preference vector beta_r
2. run batches of T tasks with R respondents and m expected contrasts/task
3. before first batch: compute + normalize sampling weights to assign baseline p_flip
4. afer each batch: pool comparisons + apply move-on rule for significant features to get updated p_flip

## 10/8: need large population pool to pull R respondents out of for each batch, no resample

In [2]:
import pandas as pd
import numpy as np
from numpy.typing import NDArray

In [3]:
import sys

In [4]:
import statsmodels.api as sm

In [5]:
from scipy.stats import norm

In [6]:
rand = np.random.default_rng(seed=42)

In [32]:
# global vars for
# - list of features/attributes
# - number of features/attributes
# - number of tasks/batch
# - size of total population
# - number of respondents/batch
# - (sigma)^2: heterogeneity in respondent preferences
# - power-law concentration
# - expected number of flipped features/task

In [7]:
FEATURES = ["setting:rural", "setting:urban", "setting:government", 
            "expression_angry", "expression_determined", "expression_sad", "expression_neutral", 
            "race_white", "race_black", "race_hispanic", "race_asian", 
            "age_old", "age_young", 
            "sign_amazon", "sign_google", "sign_microsoft", 
            "focal_single"]
P = len(FEATURES)
T = 10
POP = 3000
R = 150
RES_VAR = 0.10
gamma = 1.2
m = 2

In [8]:
# each mutually exclusive category mapped to the positions of its options, i.e. {'setting': [0, 1, 2]}
    # expects format [category]:[option] in provided list of features
CATS = {}
for i, f in enumerate(FEATURES):
    if ":" in f:
        CATS.setdefault(f.split(":", 1)[0].strip(), []).append(i)
        
# positions of binary features
BIN = [i for i, f in enumerate(FEATURES) if ":" not in f]

# positions of units that can be contrasted, 
    # i.e. [[0, 1, 2], [3], [4], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15], [16]]
UNITS = list(CATS.values()) + [[i] for i in BIN]   
N_CAT = len(CATS)
U = len(UNITS)

# reference position (first appearance of category)
REF = [cols[0] for cols in CATS.values()]      

# which columns to fit on, for a category with k options, keep k-1 columns (throw out first option)
EST = [i for i in range(P) if i not in REF]

In [9]:
CATS

{'setting': [0, 1, 2]}

In [10]:
BIN

[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

In [11]:
REF

[0]

In [12]:
EST

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

### Pool of simulated respondents

In [35]:
# get B_true baseline with index-based power-law decay: 

In [13]:
k = np.arange(1, P + 1)

In [14]:
C = 1 / np.sum(k ** (-gamma))

In [15]:
B_true = C * k ** (-gamma)

In [16]:
B_true

array([0.36088852, 0.15708585, 0.09656674, 0.06837559, 0.05231293,
       0.04203311, 0.03493457, 0.0297622 , 0.02583938, 0.02277053,
       0.02030962, 0.01829598, 0.01662038, 0.01520616, 0.01399792,
       0.01295475, 0.01204576])

In [17]:
var_matrix = RES_VAR * np.eye(P)

In [18]:
# add heterogeneity and generate full pool of respondents 

In [19]:
B = rand.multivariate_normal(mean=B_true, cov=var_matrix, size=POP)
B.shape

(3000, 17)

### "Images" (1 category with mutually exclusive features)

In [20]:
features = pd.read_csv("images/data-center-protests/feature_combinations.csv")

In [21]:
feature_cols = features.columns[1:]
features["id"] = features[feature_cols].astype(str).agg("".join, axis=1)

In [22]:
features

,id,setting_rural,setting_urban,setting_government,expression_angry,expression_determined,expression_sad,expression_neutral,race_white,race_black,race_hispanic,race_asian,age_old,age_young,sign_amazon,sign_google,sign_microsoft,focal_single
0,10000000000000000,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,10000000000000001,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,10000000000000010,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,10000000000000011,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1
4,10000000000000100,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49147,00111111111111011,0,0,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1
49148,00111111111111100,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0
49149,00111111111111101,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1
49150,00111111111111110,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0


### Helpers

In [23]:
# note that redistributing exess flip probability would lead to more contrasts than the set expected value, 
# especially as batch number increases
def flip_probs(preference):
    if (np.sum(np.abs(preference)) > 0):
        weights = np.abs(preference) / np.sum(np.abs(preference))
    else:
        weights = np.full(U, 1/U)
    p_flip = m * weights

    # cap at 1, redistribute over remaining features if exceed 
    while (p_flip > 1).any():
        pinned = p_flip >= 1
        p_flip[pinned] = 1.0
        free = ~pinned & (p_flip > 0)
        if not free.any():
            break
        p_flip[free] *= (m - pinned.sum()) / p_flip[free].sum()
    return p_flip

In [24]:
def move_on(S_b, preference):
    raw_weights = np.abs(preference).copy()
    raw_weights[list(S_b)] = 0
    return flip_probs(raw_weights)

In [25]:
def make_img2(img1, flips):
    img2 = img1.copy()
    for g in np.flatnonzero(flips):
        cols = UNITS[g]
        if g < N_CAT:                              
            current = cols[np.argmax(img1[cols])]
            img2[cols] = 0
            img2[rand.choice([c for c in cols if c != current])] = 1
        else:                                      
            img2[cols[0]] ^= 1
    return img2

In [57]:
# turning 17-feature Beta into 15-unit weights (because there are 17 columns but 15 big categories):
# use difference between highest and lowest Beta within each category
def unit_weights(preference):
    return np.array([preference[cols].max() - preference[cols].min() if g < N_CAT
                     else abs(preference[cols[0]]) for g, cols in enumerate(UNITS)])

In [63]:
# penalized logistic regression to pool new population estimates
def ridge_logit(y, X, lam):
    y = np.asarray(y, float); X = np.asarray(X, float)
    beta = np.zeros(X.shape[1])
    for _ in range(50):
        mu = 1 / (1 + np.exp(-X @ beta))
        H = X.T @ (X * (mu * (1 - mu))[:, None]) + lam * np.eye(X.shape[1])
        step = np.linalg.solve(H, X.T @ (y - mu) - lam * beta)
        beta += step
        if np.abs(step).max() < 1e-8:
            break
    mu = 1 / (1 + np.exp(-X @ beta))
    H = X.T @ (X * (mu * (1 - mu))[:, None]) + lam * np.eye(X.shape[1])
    return beta, np.sqrt(np.diag(np.linalg.inv(H)))

### Running batches (batch-adaptive, significance-gated move-on strategy)

In [60]:
# baseline example (unit flips to encompass multi-feature categories):
p_flip = flip_probs(unit_weights(B_true))
rand.binomial(1, p_flip)

array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0])

In [51]:
def run_batches(init_preferences, B_population, num_respondents, data, num_batches):
    comparisons = []
    choices = []

    # initial contrast probablities
    p_flip = flip_probs(unit_weights(init_preferences))
    
    # pull R respondents from population:
    if num_batches * num_respondents > len(B_population):
        raise ValueError(f"need {num_batches * num_respondents} respondents for "
                         f"{num_batches} batches of {num_respondents}, population has {len(B_population)}")
    batch_idx = rand.choice(len(B_population), size=(num_batches, num_respondents), replace=False)
    
    # keep track of which batch a feature first becomes significant
    tau = {}
    
    for b in range(num_batches):
        batch_respondents = B_population[batch_idx[b]]
        
        # generate T tasks for each respondent with current contrast prob
        for r in batch_respondents:
            for t in range(T):
                # for current csv setup with id column describing feature vector (i.e. '10001000101000101')
                # note no mutually exclusive features in test set (with full 2^12 combination) for simplicity 
                # realistically would need to handle when users upload data without all combinations
                img1 = np.fromiter(rand.choice(data["id"].values), dtype=int)
                flips = rand.binomial(1, p_flip)
                img2 = make_img2(img1, flips)
    
                # r "chooses" img1 or img2
                d = img1 - img2
                utility_diff = np.dot(d, r)
                p_img1 = 1 / (1 + np.exp(-utility_diff))
                choice = rand.binomial(1, p_img1)
    
                # keep track of comparisons
                comparisons.append(d[EST])
                choices.append(choice)
                
        # get pooled population estimate
        params, SE = ridge_logit(choices, comparisons, lam=1.0)
        B_pooled = np.zeros(P) 
        B_pooled[EST] = result.params

        # apply move-on rule and update
        alpha = 0.05
        critical_value = norm.ppf(1 - alpha / 2)
        #SE = result.bse
        sig = {EST[j] for j in np.where(np.abs(result.params/SE) > critical_value)[0]}
        for j in sig:
            tau.setdefault(FEATURES[j], b)
        S_b = {g for g, cols in enumerate(UNITS) if sig & set(cols)}
        p_flip = move_on(S_b, unit_weights(B_pooled))

        # print end of batch pooled comparisons, choices, preference:
        print("finished batch:", b)
        print("B_pooled:", B_pooled)
        print("S_b:", S_b)
        print("p_flip:", p_flip)
        print("sum(p_flip):", p_flip.sum())

In [52]:
x = np.fromiter(rand.choice(features["id"].values), dtype=int)
x

array([0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1])

In [50]:
y = rand.binomial(1, p_flip, size=U)
y

array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [53]:
make_img2(x, y)

array([0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1])

### Simulated Run:

In [54]:
B_true.shape

(17,)

In [55]:
B.shape

(3000, 17)

In [56]:
run_batches(B_true, B, R, features, 20)

Optimization terminated successfully.
         Current function value: 0.683573
         Iterations 4
finished batch: 0
B_pooled: [ 0.         -0.09006192 -0.281783    0.08882525 -0.02300069  0.30629751
 -0.15617494 -0.16194283  0.1162297   0.24357522  0.042487   -0.0314144
  0.25687434  0.13108031  0.09169391 -0.56631073  0.15778483]
S_b: {0, 3, 13}
p_flip: [0.         0.11834819 0.03064545 0.         0.20808296 0.21576792
 0.15486107 0.32453256 0.05660845 0.04185564 0.34225192 0.17464761
 0.1221703  0.         0.21022792]
sum(p_flip): 2.0
Optimization terminated successfully.
         Current function value: 0.687821
         Iterations 4
finished batch: 1
B_pooled: [ 0.         -0.08894939 -0.27613114 -0.10975931  0.06178702  0.31480808
 -0.11733174  0.05246401  0.11285964  0.09937712  0.0401857  -0.1012813
  0.1259141   0.02375507 -0.10106727 -0.56453715 -0.01520456]
S_b: {0, 3, 13}
p_flip: [0.         0.22843041 0.12859077 0.         0.2441901  0.10918777
 0.23488281 0.20682307 0.

Optimization terminated successfully.
         Current function value: 0.692493
         Iterations 4
finished batch: 15
B_pooled: [ 0.00000000e+00 -8.72324585e-02 -2.79530596e-01 -3.88802928e-03
  1.29848361e-02  3.08428182e-01  2.67193880e-02  5.65979165e-03
  2.96115310e-02 -1.97183927e-03  2.46352157e-02 -5.79846650e-05
  2.51104500e-02 -1.69122528e-03 -1.87098971e-01 -5.63634539e-01
  7.99982348e-03]
S_b: {0, 3, 12, 13}
p_flip: [0.         0.05541261 0.18506129 0.         0.38080761 0.08066396
 0.42202675 0.02810287 0.35110376 0.0008264  0.35787685 0.02410353
 0.         0.         0.11401435]
sum(p_flip): 2.0
Optimization terminated successfully.
         Current function value: 0.692513
         Iterations 4
finished batch: 16
B_pooled: [ 0.00000000e+00 -8.71328998e-02 -2.79503969e-01  2.96870881e-06
  1.71157551e-02  3.08513022e-01  2.00682030e-02  1.10424948e-02
  2.73297687e-02 -4.38216275e-03  3.15802908e-02  4.53509541e-05
  2.94721800e-02 -3.86338940e-03 -1.87156675e-01 -5